In [7]:
# CODE CELL 1: SNRS v1.3.5 Simulation Engine (Colab-Optimized)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import triang
from tqdm.auto import tqdm
from IPython.display import display, Markdown

# ----------------------------
# Toggles (Council Defaults)
FAST_MODE = True
REFINE_THRESHOLDS = True
PAIRED_COMPARISON = False

# Seeds
SEED_FLAT = 42
SEED_SUBCLUSTER = 123
SEED_THRESHOLD = 42

# Sampling Depth
n_sims_base = 200 if not FAST_MODE else 100
n_sims_thr_coarse = 200 if not FAST_MODE else 60
n_sims_thr_refine = 400 if not FAST_MODE else 120

# Threshold Grid
S_GRID_COARSE = np.arange(0, 8500, 500) # USD/unit/year
S_GRID_REFINE_STEP = 100 # USD/unit/year

display(Markdown(f"### SNRS v1.3.5 Run Config"))
print(f"FAST_MODE={FAST_MODE} | REFINE_THRESHOLDS={REFINE_THRESHOLDS} | PAIRED_COMPARISON={PAIRED_COMPARISON}")
print(f"n_sims_base={n_sims_base} | n_sims_thr_coarse={n_sims_thr_coarse} | n_sims_thr_refine={n_sims_thr_refine}")
print(f"SEED_FLAT={SEED_FLAT} | SEED_SUBCLUSTER={SEED_SUBCLUSTER} | SEED_THRESHOLD={SEED_THRESHOLD}")
print("Note: If PAIRED_COMPARISON=True, flat & subcluster share a paired seed scheme to reduce RNG noise.\n")

# ----------------------------
# Constants & Pricing
discount_rate = 0.03
years = 100
persons_per_unit = 1.5

baseline_energy_cost_USD_per_unit_per_year = 300.0
baseline_water_cost_USD_per_unit_per_year = 150.0
baseline_maintenance_USD_per_unit_per_year = 500.0
annual_food_cost_USD_per_person_per_year = 1200.0
baseline_fertilizer_cost_USD_per_person_per_year = 50.0

energy_price_USD_per_kwh = 0.12
water_price_USD_per_m3 = 1.5
baseline_energy_kwh_per_unit_per_year = baseline_energy_cost_USD_per_unit_per_year / energy_price_USD_per_kwh
baseline_water_m3_per_unit_per_year = baseline_water_cost_USD_per_unit_per_year / water_price_USD_per_m3

# ----------------------------
def rtri(a, m, b, rng):
    c = (m - a) / (b - a)
    return triang.rvs(c, loc=a, scale=(b - a), random_state=rng)

def discount_factors():
    t = np.arange(years + 1)
    return 1.0 / ((1.0 + discount_rate) ** t)

scenario_mult = {
    "A": {"wrapper_capex": 0.8, "hard_savings": 1.0, "eff_savings": 1.0, "policy_lag": 1.0, "participation": 1.0, "degradation": 1.0},
    "B": {"wrapper_capex": 1.0, "hard_savings": 1.2, "eff_savings": 1.1, "policy_lag": 0.9, "participation": 1.1, "degradation": 0.9},
    "C": {"wrapper_capex": 1.2, "hard_savings": 1.4, "eff_savings": 1.2, "policy_lag": 0.8, "participation": 1.2, "degradation": 0.8},
}

trust_mult = {
    "high": {"policy_lag_factor": 0.8, "aband_factor": 0.5, "participation_factor": 1.2},
    "low":  {"policy_lag_factor": 1.5, "aband_factor": 2.0, "participation_factor": 0.7},
}

# ----------------------------
def simulate_node_once(node_pop, scenario, trust, S_hard_USD_per_unit_per_year, rng,
                       overhead_policy_lag=1.0, overhead_maintenance=1.0,
                       capex_efficiency=1.0, adoption_asymptote=0.8):
    sm = scenario_mult[scenario]
    tm = trust_mult[trust]
    include_food = (scenario != "A")
    include_comp = (scenario != "A")

    p_ready = rtri(0.2, 0.3, 0.4, rng)
    p_salv  = rtri(0.3, 0.4, 0.5, rng)
    p_wo    = rtri(0.05, 0.1, 0.15, rng)
    tot = p_ready + p_salv + p_wo
    p_ready, p_salv, p_wo = p_ready/tot, p_salv/tot, p_wo/tot

    C_ready = rtri(20000, 35000, 60000, rng)
    C_salv  = rtri(50000, 80000, 120000, rng)
    R_ready = rtri(1000, 2500, 5000, rng)
    R_salv  = rtri(5000, 10000, 15000, rng)

    f_energy = rtri(0.2, 0.3, 0.4, rng)
    f_water  = rtri(0.3, 0.4, 0.5, rng)
    H_save_USD_per_person_per_year = rtri(200, 350, 500, rng)

    T80 = rtri(40, 55, 70, rng)
    lag_conv = rtri(1, 2, 3, rng)
    lag_pol0 = rtri(2, 3.5, 5, rng)
    lag_policy = lag_pol0 * tm["policy_lag_factor"] * sm["policy_lag"] * overhead_policy_lag
    lag_total = int(round(lag_conv + lag_policy))

    deg_base = rtri(0.01, 0.02, 0.03, rng)
    aband0 = rtri(0.005, 0.01, 0.02, rng)
    aband_rate = aband0 * tm["aband_factor"]

    part0 = rtri(0.1, 0.2, 0.3, rng)
    part_rate = part0 * tm["participation_factor"] * sm["participation"]
    if not include_comp: part_rate = 0.0

    retention = rtri(0.5, 0.65, 0.8, rng)
    red_maint = rtri(0.05, 0.1, 0.15, rng)
    red_deg   = rtri(0.1, 0.2, 0.3, rng)

    if include_food:
        food_offset = rtri(0.1, 0.2, 0.3, rng)
        nutrient_recovery = rtri(0.1, 0.25, 0.4, rng)
        fert_offset = rtri(0.05, 0.1, 0.2, rng)
        food_capex_USD_per_person = rtri(100, 300, 600, rng)
        food_opex_USD_per_person_per_year = rtri(20, 50, 100, rng)
    else:
        food_offset = nutrient_recovery = fert_offset = 0.0
        food_capex_USD_per_person = 0.0
        food_opex_USD_per_person_per_year = 0.0

    if include_comp:
        comp_opex_USD_per_person_per_year = rtri(10, 25, 50, rng)
        comp_offset = rtri(0.01, 0.03, 0.05, rng)
    else:
        comp_opex_USD_per_person_per_year = 0.0
        comp_offset = 0.0

    reuse_richness = rtri(0.5, 1.0, 1.5, rng)
    recoverable_share = p_ready + p_salv
    C_avg = (p_ready*C_ready + p_salv*C_salv) / recoverable_share
    R_conv_avg = (p_ready*R_ready + p_salv*R_salv) / recoverable_share

    max_units = (node_pop / persons_per_unit) * reuse_richness
    target_hh = node_pop / persons_per_unit

    t = np.arange(years + 1)
    k = 4.0 / T80
    t_mid = 0.8 * T80
    f_served = 1.0 / (1.0 + np.exp(-k * (t - t_mid)))

    effective_f = np.zeros(years + 1)
    if lag_total <= years:
        effective_f[lag_total:] = f_served[:(years + 1 - lag_total)]

    cumulative_desired = adoption_asymptote * target_hh * effective_f
    cumulative_units = np.minimum(cumulative_desired, max_units)

    occupied = np.zeros(years + 1)
    for i in range(1, years + 1):
        new_units = max(0.0, cumulative_units[i] - cumulative_units[i-1])
        occupied[i] = occupied[i-1] * (1.0 - aband_rate) + new_units

    if part_rate > 0:
        trained = part_rate * np.minimum(t / 20.0, 1.0)
        retained = trained * retention
        maint_factor = 1.0 - retained * red_maint
        deg_factor = 1.0 - retained * red_deg
    else:
        maint_factor = np.ones(years + 1)
        deg_factor = np.ones(years + 1)

    # Copilot v1.3.4 Asserts
    assert maint_factor.shape == (years + 1,)
    assert deg_factor.shape == (years + 1,)

    effective_deg = deg_base * deg_factor * sm["degradation"]
    assert effective_deg.shape == (years + 1,)

    annual_capex = np.zeros(years + 1)
    annual_benefit = np.zeros(years + 1)
    annual_maint = np.zeros(years + 1)
    annual_food_opex = np.zeros(years + 1)
    annual_comp_opex = np.zeros(years + 1)
    annual_recovery = np.zeros(years + 1)

    for i in range(1, years + 1):
        units = occupied[i]
        new_units = max(0.0, cumulative_units[i] - cumulative_units[i-1])

        new_persons = new_units * persons_per_unit
        food_capex_this_year = new_persons * food_capex_USD_per_person if include_food else 0.0
        annual_capex[i] = new_units * C_avg * sm["wrapper_capex"] * capex_efficiency + food_capex_this_year
        annual_recovery[i] = new_units * R_conv_avg

        energy_save_USD_per_unit_per_year = baseline_energy_cost_USD_per_unit_per_year * f_energy * sm["eff_savings"]
        water_save_USD_per_unit_per_year  = baseline_water_cost_USD_per_unit_per_year  * f_water  * sm["eff_savings"]
        health_save_USD_per_unit_per_year = H_save_USD_per_person_per_year * persons_per_unit

        if include_food:
            food_benefit_USD_per_unit_per_year = (
                annual_food_cost_USD_per_person_per_year * persons_per_unit * food_offset
                + baseline_fertilizer_cost_USD_per_person_per_year * persons_per_unit * nutrient_recovery * fert_offset
            )
            annual_food_opex[i] = units * (food_opex_USD_per_person_per_year * persons_per_unit)
        else:
            food_benefit_USD_per_unit_per_year = 0.0

        if include_comp:
            comp_benefit_USD_per_unit_per_year = (energy_save_USD_per_unit_per_year + water_save_USD_per_unit_per_year) * comp_offset * sm["eff_savings"]
            annual_comp_opex[i] = units * (comp_opex_USD_per_person_per_year * persons_per_unit)
        else:
            comp_benefit_USD_per_unit_per_year = 0.0

        per_unit_total_benefit = (
            (S_hard_USD_per_unit_per_year * sm["hard_savings"])
            + energy_save_USD_per_unit_per_year
            + water_save_USD_per_unit_per_year
            + health_save_USD_per_unit_per_year
            + food_benefit_USD_per_unit_per_year
            + comp_benefit_USD_per_unit_per_year
        )
        annual_benefit[i] = units * per_unit_total_benefit

        maint_base = baseline_maintenance_USD_per_unit_per_year * maint_factor[i]
        maint_with_deg = maint_base * ((1.0 + effective_deg[i]) ** i)
        annual_maint[i] = units * maint_with_deg * overhead_maintenance

    net_cf = annual_benefit - annual_capex - annual_maint - annual_food_opex - annual_comp_opex + annual_recovery
    npv_usd = float(np.sum(net_cf * discount_factors()))

    terminal_units = occupied[-1]
    terminal_energy_kwh_per_year = terminal_units * baseline_energy_kwh_per_unit_per_year * f_energy * sm["eff_savings"]
    terminal_water_m3_per_year   = terminal_units * baseline_water_m3_per_unit_per_year  * f_water  * sm["eff_savings"]

    return {
        "npv_usd": npv_usd,
        "terminal_units": terminal_units,
        "terminal_energy_kwh_per_year": terminal_energy_kwh_per_year,
        "terminal_water_m3_per_year": terminal_water_m3_per_year,
        "material_recovery_usd_total": float(np.sum(annual_recovery)),
        "food_self_provision_percent": float(food_offset * 100.0) if include_food else 0.0,
    }

# ----------------------------
def simulate_node(node_pop, scenario, trust, S_hard_USD_per_unit_per_year, n_sims, seed, **kwargs):
    rng = np.random.default_rng(seed)
    npvs, water, energy, recovery, food = [], [], [], [], []
    for _ in range(n_sims):
        out = simulate_node_once(node_pop, scenario, trust, S_hard_USD_per_unit_per_year, rng, **kwargs)
        npvs.append(out["npv_usd"])
        water.append(out["terminal_water_m3_per_year"])
        energy.append(out["terminal_energy_kwh_per_year"])
        recovery.append(out["material_recovery_usd_total"])
        food.append(out["food_self_provision_percent"])
    npvs = np.array(npvs)
    return {
        "npv_median_MUSD": np.median(npvs) / 1e6,
        "npv_p5_MUSD": np.percentile(npvs, 5) / 1e6,
        "npv_p95_MUSD": np.percentile(npvs, 95) / 1e6,
        "prob_positive": float(np.mean(npvs > 0)),
        "terminal_water_Mm3_per_year_median": np.median(water) / 1e6,
        "terminal_energy_TWh_per_year_median": np.median(energy) / 1e9,
        "material_recovery_MUSD_median": np.median(recovery) / 1e6,
        "food_self_provision_percent_median": float(np.median(food)),
    }

def simulate_subcluster(scenario, trust, S_hard_USD_per_unit_per_year, n_sims, seed,
                        overhead_policy=1.1, overhead_maintenance=1.1, capex_efficiency=0.95):
    rng = np.random.default_rng(seed)
    npvs, water, energy, recovery, food = [], [], [], [], []
    for _ in range(n_sims):
        npv_s = w_s = e_s = r_s = 0.0
        food_v = []
        for __ in range(5):
            out = simulate_node_once(20000, scenario, trust, S_hard_USD_per_unit_per_year, rng,
                overhead_policy_lag=overhead_policy,
                overhead_maintenance=overhead_maintenance,
                capex_efficiency=capex_efficiency
            )
            npv_s += out["npv_usd"]
            w_s   += out["terminal_water_m3_per_year"]
            e_s   += out["terminal_energy_kwh_per_year"]
            r_s   += out["material_recovery_usd_total"]
            food_v.append(out["food_self_provision_percent"])
        npvs.append(npv_s); water.append(w_s); energy.append(e_s); recovery.append(r_s)
        food.append(float(np.mean(food_v)))
    npvs = np.array(npvs)
    return {
        "npv_median_MUSD": np.median(npvs) / 1e6,
        "npv_p5_MUSD": np.percentile(npvs, 5) / 1e6,
        "npv_p95_MUSD": np.percentile(npvs, 95) / 1e6,
        "prob_positive": float(np.mean(npvs > 0)),
        "terminal_water_Mm3_per_year_median": np.median(water) / 1e6,
        "terminal_energy_TWh_per_year_median": np.median(energy) / 1e9,
        "material_recovery_MUSD_median": np.median(recovery) / 1e6,
        "food_self_provision_percent_median": float(np.median(food)),
    }

# ----------------------------
def _find_crossing(S_grid, med_npvs):
    crossing = np.nan
    for i in range(len(S_grid) - 1):
        n1, n2 = med_npvs[i], med_npvs[i+1]
        if n1 == 0.0: return float(S_grid[i])    # Copilot exact-zero shortcut
        if n2 == 0.0: return float(S_grid[i+1])

        if n1 < 0.0 and n2 > 0.0:
            S1, S2 = float(S_grid[i]), float(S_grid[i+1])
            crossing = S1 - n1*(S2-S1)/(n2-n1)
            break
    return crossing

def find_threshold_flat(node_pop, scenario, trust, S_grid, n_sims, seed):
    med_npvs = []
    for S in S_grid:
        res = simulate_node(node_pop, scenario, trust, S, n_sims=n_sims, seed=seed)
        med_npvs.append(res["npv_median_MUSD"])
    crossing_unit = _find_crossing(S_grid, med_npvs)
    crossing_person = crossing_unit / persons_per_unit if not np.isnan(crossing_unit) else np.nan
    return crossing_unit, crossing_person, med_npvs

def find_threshold_subcluster(scenario, trust, S_grid, n_sims, seed):
    med_npvs = []
    for S in S_grid:
        res = simulate_subcluster(scenario, trust, S, n_sims=n_sims, seed=seed)
        med_npvs.append(res["npv_median_MUSD"])
    crossing_unit = _find_crossing(S_grid, med_npvs)
    crossing_person = crossing_unit / persons_per_unit if not np.isnan(crossing_unit) else np.nan
    return crossing_unit, crossing_person, med_npvs

def refine_threshold_around_crossing(find_fn, coarse_S, coarse_med, step, n_sims_refine):
    idx = None
    for i in range(len(coarse_S) - 1):
        if coarse_med[i] <= 0.0 and coarse_med[i+1] >= 0.0:
            idx = i
            break
    if idx is None:
        return np.nan, np.nan

    lo = float(coarse_S[idx])
    hi = float(coarse_S[idx+1])
    refined_grid = np.arange(lo, hi + step, step)

    cross_u, cross_p, _ = find_fn(refined_grid, n_sims_refine)
    return cross_u, cross_p

# ----------------------------
# Runner with Colab tqdm Progress Bars
node_sizes = [5000, 10000, 50000, 100000, 250000, 1000000]
scenarios = ["A", "B", "C"]
trusts = ["high", "low"]

rows, thr_rows = [], []

print("Running Monte Carlo sweeps...")

# Baselines + thresholds (flat)
for pop in tqdm(node_sizes, desc="Standard Nodes"):
    for sc in scenarios:
        for tr in trusts:
            active_seed = SEED_FLAT if PAIRED_COMPARISON else SEED_THRESHOLD

            # Coarse threshold
            thr_u, thr_p, med = find_threshold_flat(pop, sc, tr, S_GRID_COARSE, n_sims_thr_coarse, active_seed)

            # Optional refine
            if REFINE_THRESHOLDS and not np.isnan(thr_u):
                def _find_fn(ref_grid, sims): return find_threshold_flat(pop, sc, tr, ref_grid, sims, active_seed)
                thr_u2, thr_p2 = refine_threshold_around_crossing(_find_fn, S_GRID_COARSE, med, S_GRID_REFINE_STEP, n_sims_thr_refine)
                if not np.isnan(thr_u2):
                    thr_u, thr_p = thr_u2, thr_p2

            thr_rows.append({
                "node_pop": pop, "scenario": sc, "trust": tr,
                "min_S_hard_USD_per_unit_per_year": thr_u,
                "min_S_hard_USD_per_person_per_year": thr_p,
            })

            base = simulate_node(pop, sc, tr, 3500, n_sims_base, SEED_FLAT)
            base.update({"node_pop": pop, "scenario": sc, "trust": tr})
            rows.append(base)

# Baselines + thresholds (subcluster)
for sc in tqdm(scenarios, desc="Subcluster 5x20k"):
    for tr in trusts:
        active_seed = SEED_FLAT if PAIRED_COMPARISON else SEED_SUBCLUSTER

        thr_u, thr_p, med = find_threshold_subcluster(sc, tr, S_GRID_COARSE, n_sims_thr_coarse, active_seed)

        if REFINE_THRESHOLDS and not np.isnan(thr_u):
            def _find_fn(ref_grid, sims): return find_threshold_subcluster(sc, tr, ref_grid, sims, active_seed)
            thr_u2, thr_p2 = refine_threshold_around_crossing(_find_fn, S_GRID_COARSE, med, S_GRID_REFINE_STEP, n_sims_thr_refine)
            if not np.isnan(thr_u2):
                thr_u, thr_p = thr_u2, thr_p2

        thr_rows.append({
            "node_pop": "100k_subcluster(5x20k)", "scenario": sc, "trust": tr,
            "min_S_hard_USD_per_unit_per_year": thr_u,
            "min_S_hard_USD_per_person_per_year": thr_p,
        })

        base = simulate_subcluster(sc, tr, 3500, n_sims_base, SEED_SUBCLUSTER)
        base.update({"node_pop": "100k_subcluster(5x20k)", "scenario": sc, "trust": tr})
        rows.append(base)

df = pd.DataFrame(rows)
thr_df = pd.DataFrame(thr_rows)

display(Markdown("### Baseline Results (S_hard = $3500/unit/year)"))
display(df.round(3))

display(Markdown("### Threshold Results (Median NPV >= 0)"))
display(Markdown("*(Per-person is expected to be scale-invariant in Phase‑1 linear model.)*"))
display(thr_df.round(2))

# Plotting
plt.figure(figsize=(12, 8))
plot_df = thr_df[~thr_df["node_pop"].astype(str).str.contains("subcluster")].copy()
plot_df["node_pop"] = plot_df["node_pop"].astype(float)

for sc in scenarios:
    for tr in trusts:
        subset = plot_df[(plot_df["scenario"] == sc) & (plot_df["trust"] == tr)]
        plt.plot(
            subset["node_pop"],
            subset["min_S_hard_USD_per_person_per_year"],
            marker="o",
            label=f"Scenario {sc}, {tr}-trust")

plt.xscale("log")
plt.xlabel("Node Population")
plt.ylabel("Minimum S_hard (USD/person/year)")
plt.title("Threshold Hard Savings Required for Positive NPV (Phase-1 Linear Model)")
plt.legend()
plt.grid(True)
plt.show()

print("Note: Flat horizontal lines are expected under Phase-1 linear assumptions (scale invariance).")

### SNRS v1.3.5 Run Config

FAST_MODE=True | REFINE_THRESHOLDS=True | PAIRED_COMPARISON=False
n_sims_base=100 | n_sims_thr_coarse=60 | n_sims_thr_refine=120
SEED_FLAT=42 | SEED_SUBCLUSTER=123 | SEED_THRESHOLD=42
Note: If PAIRED_COMPARISON=True, flat & subcluster share a paired seed scheme to reduce RNG noise.

Running Monte Carlo sweeps...


Standard Nodes:   0%|          | 0/6 [00:00<?, ?it/s]

Subcluster 5x20k:   0%|          | 0/3 [00:00<?, ?it/s]

KeyboardInterrupt: 